In [1]:
import pandas as pd
import openai
import pinecone
from langchain.text_splitter import CharacterTextSplitter
import itertools
from tqdm import tqdm

PINECONE_ENVIRONMENT = "gcp-starter"

index_name = "law-gpt"

df = pd.read_csv("processed_data.csv")

def batch_embeddings(texts, batch_size=10):
    for i in range(0, len(texts), batch_size):
        response = openai.Embedding.create(input=texts[i:i + batch_size], engine="text-embedding-ada-002", api_key = OPENAI_API_KEY)
        embeddings = [item['embedding'] for item in response['data']]
        yield embeddings

# Helper function to break the data into batches and return as a list
def create_batches(data, batch_size=100):
    """Create and return a list of batch-sized chunks from data."""
    return [data[i:i + batch_size] for i in range(0, len(data), batch_size)]

/home/veleref/anaconda3/envs/legal_assistant_bot/lib/python3.10/site-packages/pinecone/index.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=3000)


# Initialize Pinecone
pinecone.init(api_key=PINECONE_API_KEY, environment=PINECONE_ENVIRONMENT)
if index_name not in pinecone.list_indexes():
    pinecone.create_index(index_name, dimension=1536)  # Ensure the dimension is correct
index = pinecone.Index(index_name)

# New variable for upload threshold
upload_threshold = 5 # Set this to the number of rows after which to upload

# Prepare data with metadata
data_to_upload = []
processed_rows = 0  # Tracks the number of processed rows

for _, row in tqdm(df.iterrows(), desc="Processing rows", total=len(df)):
    chunks = text_splitter.split_text(row['text'])
    for chunk_index, chunk in enumerate(chunks):
        chunk_embedding = next(batch_embeddings([chunk]))
        metadata = {'text': chunk, 'original_id': row['id']}
        data_to_upload.append((f"{row['id']}-{chunk_index}", chunk_embedding[0], metadata))
    
    processed_rows += 1
    if processed_rows >= upload_threshold:
        # Upload the accumulated data
        # batches = create_batches(data_to_upload, batch_size=40)
        # for batch in batches:
        index.upsert(vectors=data_to_upload)
        
        # Reset variables
        data_to_upload = []
        processed_rows = 0

Processing rows:  99%|█████████▉| 989/1000 [23:57<01:24,  7.71s/it]Created a chunk of size 3478, which is longer than the specified 3000
Created a chunk of size 3269, which is longer than the specified 3000
Created a chunk of size 3473, which is longer than the specified 3000
Processing rows: 100%|██████████| 1000/1000 [24:47<00:00,  1.49s/it]


In [12]:
# Generate the query vector
query_text = "tax evasion and drug crimes"
top_k = 1
min_text_length = 1


def query_and_process_results(query_text, min_text_length=1, top_k=3):
    """
    Query the Pinecone index with the given text and process the results.

    Args:
    query_text (str): The text to query.
    min_text_length (int): Minimum length of text to include in results.
    top_k (int): Number of top results to retrieve.

    Returns:
    None
    """
    # Generate the query vector
    query_vector = next(batch_embeddings([query_text]))[0]

    # Perform the query and request metadata
    query_results = index.query(vector=query_vector, top_k=top_k, include_metadata=True)

    results = []
    for result in query_results["matches"]:
        if 'metadata' in result and 'text' in result['metadata'] and len(result['metadata']['text']) > min_text_length:
            results.append({
                "id": result['id'],
                "score": result['score'],
                "text": result['metadata']['text']
            })
    return results

query_and_process_results(query_text, top_k=2)

[{'id': '1552778-0',
  'score': 0.801048398,
  'text': 'OPINION\nMINZNER, Chief Judge.\nDefendant appeals from a judgment entered after a jury trial convicting him of one count of trafficking in cocaine, a second-degree felony, contrary to NMSA 1978, Sections 30-31-20(A)(2) and (B)(1) (Repl. Pamp.1989), and one count of conspiracy to traffic cocaine, a third-degree felony, contrary to NMSA 1978, Sections 30-28-2 (Repl. Pamp.1984) and 30-31-20. On appeal, Defendant raises four issues: (1) the jury was not instructed properly on his entrapment defense; (2) he was entitled to a directed verdict on the trafficking count because he established entrapment as a matter of law; (3) there was insufficient evidence to support the conspiracy-to-traffic count; and (4) he was denied a fair trial by prosecutorial misconduct during closing argument. We affirm.\nFACTS\nAt trial Kel Smith testified that he had five felony convictions for check forgery and had been in prison four times. He became a polic